# ============================================================
# BENCHMARK METHOD 4: RandAR Baseline (Table 3 Reproduction)
# Decoder-only Autoregressive Visual Generation in Random Orders (CVPR 2025 Oral)
# Flat Vocabulary (N = 16,384) | 10-Class ImageNet Evaluation
# Tối ưu hóa siêu tốc: Tensor Core FP16 (AMP) + Zero-latency Preloaded Dataset
# ============================================================


In [ ]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone các Repositories cần thiết
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR          = '/kaggle/working'
RANDAR_DIR        = f'{WORK_DIR}/RandAR'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_RANDAR_DIR   = f'{WORK_DIR}/checkpoints_randar'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_RANDAR_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)

print('Cài đặt dependencies cho RandAR & LlamaGen Evaluator...')
subprocess.run('pip install -q omegaconf timm safetensors einops huggingface_hub pandas tabulate scipy', shell=True)

def clone_or_pull(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã tồn tại, git pull cập nhật...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_or_pull('https://github.com/ziqipang/RandAR.git', RANDAR_DIR)
clone_or_pull('https://github.com/FoundationVision/LlamaGen.git', LLAMA_DIR)
clone_or_pull('https://github.com/nta2112/Masce-baseline-benchmark.git', BENCHMARK_DIR)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


In [ ]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của RandAR & VQ-VAE (LlamaGen Tokenizer)
# ============================================================
from huggingface_hub import hf_hub_download
import os

# 1. VQ-VAE Tokenizer (Chung chuẩn với LlamaGen: VQ-DS16, codebook 16,384)
VQ_PATH = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
if not os.path.exists(VQ_PATH):
    print('Đang tải Tokenizer vq_ds16_c2i.pt...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename='vq_ds16_c2i.pt', local_dir=WEIGHTS_DIR)
    print('✅ Đã tải: vq_ds16_c2i.pt')
else:
    print('✅ Tokenizer vq_ds16_c2i.pt đã có sẵn.')

# 2. Checkpoint Pretrained của RandAR-L (0.3B ~ 343M params)
RANDAR_CKPT_NAME = 'randar_0.3b_llamagen_360k_bs_1024_lr_0.0004.safetensors'
RANDAR_CKPT_PATH = os.path.join(WEIGHTS_DIR, RANDAR_CKPT_NAME)
if not os.path.exists(RANDAR_CKPT_PATH):
    print(f'Đang tải Pretrained RandAR-L weights ({RANDAR_CKPT_NAME})...')
    hf_hub_download(repo_id='ziqipang/RandAR', filename=RANDAR_CKPT_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {RANDAR_CKPT_NAME}')
else:
    print(f'✅ RandAR checkpoint đã có sẵn: {RANDAR_CKPT_PATH}')


In [ ]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu Huấn luyện & Reference Batch 10-Class
# ĐÚNG 10 CLASS (Classes 0..9) x 1,000 ảnh = 10,000 ảnh chuẩn
# ============================================================
import os, glob, io, tarfile, time, shutil, subprocess
import numpy as np, torch
from pathlib import Path
from huggingface_hub import snapshot_download

REF_NPZ = f'{WORK_DIR}/ref_10class_balanced_1000.npz'
labels_path = f'{CODES_DIR}/labels.npy'

# Tìm kiếm Reference Batch có sẵn trong Kaggle input để tiết kiệm thời gian
REF_INPUT_CANDIDATES = [
    REF_NPZ,
    '/kaggle/input/ref-10class-balanced-1000/ref_10class_balanced_1000.npz',
] + glob.glob('/kaggle/input/**/ref_10class_balanced_1000.npz', recursive=True)

existing_ref = next((p for p in REF_INPUT_CANDIDATES if os.path.exists(p) and os.path.getsize(p) > 1000000), None)
if existing_ref and not os.path.exists(REF_NPZ):
    print(f'✅ Tìm thấy Reference Batch có sẵn trong input: {existing_ref}')
    shutil.copy(existing_ref, REF_NPZ)
    print(f'   -> Đã sao chép sang: {REF_NPZ} ({os.path.getsize(REF_NPZ)/1e6:.1f} MB)')

# Trích xuất 10,000 samples token codes nếu chưa có
existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    os.makedirs(CODES_DIR, exist_ok=True)
    print('Đang tải imagenet-llamagen-cache để lấy đúng 10,000 ảnh huấn luyện (10 class x 1,000 ảnh)...')
    snapshot_download(
        repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
        repo_type='dataset', local_dir=WDS_DIR,
        ignore_patterns=['*.json', '*.yaml', 'README*'],
    )
    tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))
    if not tar_files:
        tar_files = sorted(Path('/kaggle/input').glob('**/*.tar'))
    assert len(tar_files) > 0, 'Không tìm thấy file tar của dataset!'
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')

    TARGET_SYNSETS = {
        'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4,
        'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9,
    }
    MAX_PER_CLASS = 1000
    class_counts = {c: 0 for c in range(10)}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            synset = parts[-2]
            label = TARGET_SYNSETS.get(synset, -1)
            if label == -1 or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples token codes trong {CODES_DIR}!')
else:
    print(f'✅ Đã có sẵn 10,000 samples token codes trong {CODES_DIR}!')

# Tạo Reference Batch (1,000 ảnh thật cân bằng 10 class) nếu chưa có
if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)
    all_labels = np.load(labels_path)

    ref_indices = []
    for c in range(10):
        c_idxs = np.where(all_labels == c)[0][:100]
        ref_indices.extend(c_idxs)
    ref_indices = np.array(ref_indices)
    ref_codes = all_codes[ref_indices]

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')


In [ ]:
# ============================================================
# CELL 4 — HUẤN LUYỆN 10 EPOCHS CHO RANDAR BASELINE (N=16,384)
# Hỗ trợ tự động nhận diện Checkpoint có sẵn để BỎ QUA huấn luyện
# ============================================================
import subprocess, os, sys, shutil, glob

# Kiểm tra xem người dùng đã tải lên checkpoint có sẵn hay chưa
CKPT_CANDIDATES = [
    # Checkpoint từ Kaggle Input (Dataset/Models)
    '/kaggle/input/randar-checkpoint-10class/ckpt_randar_10class_final.pt',
    '/kaggle/input/randar-checkpoint-10class/ckpt_randar_epoch_10.pt',
    # Checkpoint trong thư mục working nếu đã có từ trước
    f'{CKPT_RANDAR_DIR}/ckpt_randar_10class_final.pt',
    f'{CKPT_RANDAR_DIR}/ckpt_randar_epoch_10.pt',
] + glob.glob('/kaggle/input/**/ckpt_randar*.pt', recursive=True) \
  + glob.glob('/kaggle/input/**/*randar*10*.pt', recursive=True) \
  + glob.glob('/kaggle/input/**/*randar*final*.pt', recursive=True)

EXISTING_CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p) and os.path.isfile(p)), None)

if EXISTING_CKPT:
    print('=' * 70)
    print(f'✅ PHÁT HIỆN CHECKPOINT RANDAR BASELINE 10 EPOCHS CÓ SẴN:')
    print(f'   -> Đường dẫn: {EXISTING_CKPT}')
    print(f'   -> Dung lượng: {os.path.getsize(EXISTING_CKPT) / 1e6:.1f} MB')
    print('=' * 70)
    print('⏩ BỎ QUA QUÁ TRÌNH HUẤN LUYỆN!')
    print('👉 Chuyển thẳng sang CELL 5 để sinh 1,000 ảnh mẫu và CELL 6 để đánh giá Metrics.')
else:
    TRAIN_SCRIPT = f'{WORK_DIR}/train_randar.py'

    train_code = '''#!/usr/bin/env python3
"""
Fine-tuning RandAR Baseline (CVPR 2025 Oral) on 10-Class ImageNet.
Flat vocabulary N = 16,384, Random-order Autoregressive generation.
Optimized with Pure Float32 Precision (Zero NaN, 100% Numerically Stable on Tesla T4).
Configured for 10.0 Epochs (1,560 steps total), Fast Dataset & Efficient Memory.
"""
import os, sys, time, argparse, glob, types
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP
from omegaconf import OmegaConf

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

class FastRandARTokenDataset(Dataset):
    def __init__(self, codes_dir):
        shards = sorted(glob.glob(f"{codes_dir}/shard_*.npy"))
        codes_np = np.concatenate([np.load(s) for s in shards], axis=0)
        labels_np = np.load(f"{codes_dir}/labels.npy")
        assert len(codes_np) == len(labels_np)
        self.codes = torch.from_numpy(codes_np).long()
        self.labels = torch.from_numpy(labels_np).long()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        return self.codes[idx], self.labels[idx]

def forward_train_stable(self, idx, cond_idx, token_order=None, input_pos=None, targets=None, mask=None, valid=None):
    bs = idx.shape[0]
    if token_order is None:
        token_order = torch.arange(self.block_size, device=self.tok_embeddings.weight.device, dtype=torch.long).unsqueeze(0).repeat(bs, 1)

    idx = torch.gather(idx.unsqueeze(-1), 1, token_order.unsqueeze(-1)).squeeze(-1).contiguous()
    targets = torch.gather(targets.unsqueeze(-1), 1, token_order.unsqueeze(-1)).squeeze(-1).contiguous()

    self.freqs_cis = self.freqs_cis.to(cond_idx.device)
    cond_embeddings = self.cls_embedding(cond_idx, train=self.training)[:, : self.cls_token_num]
    token_embeddings = self.tok_embeddings(idx)
    token_embeddings = self.tok_dropout(token_embeddings)
    position_instruction_tokens = self.get_position_instruction_tokens(token_order)

    from RandAR.model.utils import interleave_tokens
    from torch.utils.checkpoint import checkpoint

    h = torch.cat((cond_embeddings, interleave_tokens(position_instruction_tokens, token_embeddings)), dim=1)
    token_freqs_cis = self.freqs_cis[self.cls_token_num:].clone().to(token_order.device)[token_order]
    freqs_cis = torch.cat((self.freqs_cis[:self.cls_token_num].unsqueeze(0).repeat(bs, 1, 1, 1), interleave_tokens(token_freqs_cis, token_freqs_cis)), dim=1)

    for layer in self.layers:
        if self.grad_checkpointing:
            h = checkpoint(layer, h, freqs_cis, input_pos, mask, use_reentrant=False)
        else:
            h = layer(h, freqs_cis, input_pos, mask)

    h = self.norm(h)
    logits = self.output(h)
    token_logits = logits[:, self.cls_token_num::2].contiguous()
    token_logits = torch.clamp(token_logits, min=-50.0, max=50.0)
    loss = F.cross_entropy(token_logits.view(-1, token_logits.size(-1)), targets.view(-1))
    return token_logits, loss, token_order

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--randar-repo", type=str, required=True)
    parser.add_argument("--vq-ckpt", type=str, required=True)
    parser.add_argument("--pretrained", type=str, required=True)
    parser.add_argument("--codes-dir", type=str, required=True)
    parser.add_argument("--epochs", type=float, default=10.0)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--grad-accum", type=int, default=2)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--min-lr", type=float, default=1e-6)
    parser.add_argument("--warmup-steps", type=int, default=100)
    parser.add_argument("--grad-checkpointing", action="store_true", default=False)
    parser.add_argument("--out-dir", type=str, default="checkpoints_randar")
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--log-every", type=int, default=50)
    args = parser.parse_args()

    if args.randar_repo not in sys.path:
        sys.path.insert(0, args.randar_repo)
    from RandAR.util import instantiate_from_config, load_safetensors

    is_dist = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if is_dist:
        torch.distributed.init_process_group(backend="nccl")
        rank = int(os.environ["RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
    else:
        rank, world_size, local_rank = 0, 1, 0
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    is_master = (rank == 0)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    cfg_path = os.path.join(args.randar_repo, "configs/randar/randar_l_0.3b_llamagen.yaml")
    config = OmegaConf.load(cfg_path)
    config.ar_model.params.grad_checkpointing = args.grad_checkpointing
    model = instantiate_from_config(config.ar_model).to(device)

    model.forward_train = types.MethodType(forward_train_stable, model)

    if args.pretrained.endswith(".safetensors"):
        weights = load_safetensors(args.pretrained)
    else:
        st = torch.load(args.pretrained, map_location="cpu")
        weights = st.get("model", st)
    model.load_state_dict(weights, strict=False)

    if is_master:
        print(f"[RandAR Baseline] Loaded pretrained weights from {args.pretrained}")

    if is_dist:
        model = DDP(model, device_ids=[local_rank], find_unused_parameters=False)

    dataset = FastRandARTokenDataset(args.codes_dir)
    sampler = torch.utils.data.distributed.DistributedSampler(dataset, shuffle=True) if is_dist else None
    loader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler, shuffle=(sampler is None),
                        num_workers=2, pin_memory=True, drop_last=True)

    try:
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.05, fused=True)
    except Exception:
        optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.05)

    steps_per_epoch = len(loader) // args.grad_accum
    total_steps = int(args.epochs * steps_per_epoch)
    if is_master:
        print(f"[RandAR Baseline] Samples: {len(dataset):,} | Steps/epoch: {steps_per_epoch} | Total: {total_steps} (Pure Float32, BS={args.batch_size}x{args.grad_accum}, Checkpointing={args.grad_checkpointing})")

    global_step = 0
    t0 = time.time()
    step_t0 = time.time()
    os.makedirs(args.out_dir, exist_ok=True)

    model.train()
    optimizer.zero_grad(set_to_none=True)
    accum_idx = 0

    while global_step < total_steps:
        if sampler is not None:
            sampler.set_epoch(int(global_step // max(1, steps_per_epoch)))

        for tokens, labels in loader:
            if global_step >= total_steps:
                break

            tokens = tokens.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            token_order = torch.rand(tokens.size(0), 256, device=device).argsort(dim=-1)

            if global_step < args.warmup_steps:
                lr = args.min_lr + (args.lr - args.min_lr) * (global_step / max(args.warmup_steps, 1))
            else:
                progress = (global_step - args.warmup_steps) / max(total_steps - args.warmup_steps, 1)
                lr = args.min_lr + 0.5 * (args.lr - args.min_lr) * (1.0 + np.cos(np.pi * progress))
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            logits, loss, _ = model(tokens, labels, token_order=token_order, targets=tokens)
            loss_scaled = loss / args.grad_accum

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[Warning] Loss became {loss.item()} at step {global_step}, skipping...")
                optimizer.zero_grad(set_to_none=True)
                accum_idx = 0
                continue

            loss_scaled.backward()
            accum_idx += 1

            if accum_idx % args.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                accum_idx = 0

                if is_master and (global_step % args.log_every == 0 or global_step == total_steps):
                    elapsed = time.time() - t0
                    step_time = (time.time() - step_t0) / args.log_every
                    step_t0 = time.time()
                    print(f"[RandAR] step {global_step}/{total_steps} | loss={loss.item():.4f} | lr={lr:.2e} | {step_time*1000:.1f}ms/step | elapsed: {elapsed:.0f}s", flush=True)

                if is_master and args.save_every > 0 and (global_step % (steps_per_epoch * args.save_every) == 0):
                    epoch_cur = int(global_step // steps_per_epoch)
                    save_path = os.path.join(args.out_dir, f"ckpt_randar_epoch_{epoch_cur}.pt")
                    raw_model = model.module if hasattr(model, "module") else model
                    torch.save({"model": raw_model.state_dict(), "epoch": epoch_cur, "step": global_step}, save_path)
                    print(f"💾 Epoch {epoch_cur} Checkpoint saved to: {save_path}")

    if is_master:
        final_path = os.path.join(args.out_dir, "ckpt_randar_10class_final.pt")
        raw_model = model.module if hasattr(model, "module") else model
        torch.save({"model": raw_model.state_dict(), "epoch": int(args.epochs), "step": global_step}, final_path)
        print(f"✅ Final Checkpoint saved to: {final_path}")

if __name__ == "__main__":
    main()
'''

    with open(TRAIN_SCRIPT, 'w', encoding='utf-8') as f:
        f.write(train_code)

    if os.path.exists(f'{BENCHMARK_DIR}/baselines/04_randar'):
        shutil.copy(TRAIN_SCRIPT, f'{BENCHMARK_DIR}/baselines/04_randar/train_randar.py')
        if os.path.exists(f'{BENCHMARK_DIR}/baselines/04_randar/randar_infer.py'):
            shutil.copy(f'{BENCHMARK_DIR}/baselines/04_randar/randar_infer.py', f'{WORK_DIR}/randar_infer.py')

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29520',
        TRAIN_SCRIPT,
        '--randar-repo',   RANDAR_DIR,
        '--vq-ckpt',       f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
        '--pretrained',    RANDAR_CKPT_PATH,
        '--codes-dir',     CODES_DIR,
        '--epochs',        '10.0',
        '--batch-size',    '16',
        '--grad-accum',    '2',
        '--lr',            '1e-4',
        '--min-lr',        '1e-6',
        '--out-dir',       CKPT_RANDAR_DIR,
        '--save-every',    '5',
        '--log-every',     '50'
    ]

    print('Bắt đầu HUẤN LUYỆN 10 Epochs cho RandAR Baseline (Random-order AR, N=16,384) trên 2x GPU T4...')
    print('⚡ Kích hoạt Pure Float32 Precision (Zero NaN, 100% Ổn định số học, BS=16x2)...')
    print(' '.join(cmd))
    print('=' * 70)

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode == 0:
        print('\n✅ HUẤN LUYỆN 10 EPOCHS CHO RANDAR BASELINE HOÀN TẤT THÀNH CÔNG!')
    else:
        print('\n❌ Lỗi trong quá trình huấn luyện!')


In [ ]:
# ============================================================
# CELL 5 — Sinh 1,000 Ảnh Mẫu từ RandAR Baseline
# Sinh chuẩn 100 ảnh/class cho ĐÚNG 10 CLASS (Classes 0..9)
# ============================================================
import subprocess, os, sys, glob, shutil

INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/04_randar/randar_infer.py'
if not os.path.exists(INFER_SCRIPT):
    INFER_SCRIPT = f'{WORK_DIR}/randar_infer.py'

SAMPLES_OUT = f'{WORK_DIR}/samples_randar_baseline_1000.npz'

# Ưu tiên load Checkpoint người dùng đã có từ /kaggle/input hoặc final checkpoint
CKPT_CANDIDATES = [
    '/kaggle/input/randar-checkpoint-10class/ckpt_randar_10class_final.pt',
    '/kaggle/input/randar-checkpoint-10class/ckpt_randar_epoch_10.pt',
    f'{CKPT_RANDAR_DIR}/ckpt_randar_10class_final.pt',
    f'{CKPT_RANDAR_DIR}/ckpt_randar_epoch_10.pt',
] + glob.glob('/kaggle/input/**/ckpt_randar*.pt', recursive=True) \
  + glob.glob('/kaggle/input/**/*randar*10*.pt', recursive=True) \
  + glob.glob('/kaggle/input/**/*randar*final*.pt', recursive=True) \
  + sorted(glob.glob(f'{CKPT_RANDAR_DIR}/ckpt_randar_epoch_*.pt'))

CKPT_USE = next((p for p in CKPT_CANDIDATES if os.path.exists(p) and os.path.isfile(p)), None)

if CKPT_USE:
    print('=' * 70)
    print(f'✅ Sử dụng Checkpoint RandAR Baseline: {CKPT_USE}')
    print(f'   -> Dung lượng: {os.path.getsize(CKPT_USE) / 1e6:.1f} MB')
    print('=' * 70)
else:
    raise FileNotFoundError(f'[FATAL ERROR] Không tìm thấy checkpoint nào trong /kaggle/input hoặc {CKPT_RANDAR_DIR}!')

cmd = [
    'python', INFER_SCRIPT,
    '--randar-repo',  RANDAR_DIR,
    '--vq-ckpt',      f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
    '--ckpt',         CKPT_USE,
    '--num-samples',  '1000',
    '--batch-size',   '16',
    '--cfg',          '3.0',
    '--top-k',        '1000',
    '--top-p',        '1.0',
    '--steps',        '88',
    '--out',          SAMPLES_OUT
]

print('Bắt đầu sinh 1,000 ảnh RandAR Baseline trên 2x GPU T4 (10 class cân bằng)...')
print(' '.join(cmd))
print('=' * 70)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ ĐÃ SINH XONG 1,000 ẢNH TỪ RANDAR BASELINE LƯU TẠI: {SAMPLES_OUT}')
else:
    print('\n❌ Lỗi trong quá trình sinh ảnh!')


In [ ]:
# ============================================================
# CELL 6 — ĐÁNH GIÁ METRICS: FID, IS, Precision, Recall
# So sánh sòng phẳng trên cùng Reference Batch ref_10class_balanced_1000.npz
# ============================================================
import subprocess, os, sys, json, re

SAMPLES_OUT = f'{WORK_DIR}/samples_randar_baseline_1000.npz'
REF_NPZ     = f'{WORK_DIR}/ref_10class_balanced_1000.npz'

eval_cmd = f'CUDA_VISIBLE_DEVICES=1 python {LLAMA_DIR}/evaluations/c2i/evaluator.py {REF_NPZ} {SAMPLES_OUT}'
print('Khởi chạy LlamaGen Evaluator trên GPU 1 (dành trọn VRAM cho TF Inception-v3):')
print(eval_cmd)
print('=' * 70)

proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
full_output = []
for line in proc.stdout:
    print(line, end='', flush=True)
    full_output.append(line)
proc.wait()

eval_text = "".join(full_output)
metrics = {}
for line in eval_text.split('\n'):
    if 'FID' in line and ':' in line:
        try: metrics['fid'] = float(line.split(':')[-1].strip().split()[0])
        except Exception: pass
    if 'Inception Score' in line and ':' in line:
        try: metrics['is'] = float(line.split(':')[-1].strip().split()[0])
        except Exception: pass
    if 'Precision' in line and ':' in line:
        try: metrics['precision'] = float(line.split(':')[-1].strip().split()[0])
        except Exception: pass
    if 'Recall' in line and ':' in line:
        try: metrics['recall'] = float(line.split(':')[-1].strip().split()[0])
        except Exception: pass

with open(f'{WORK_DIR}/randar_baseline_metrics.json', 'w') as f:
    json.dump(metrics, f)

print('\n✅ HOÀN TẤT ĐÁNH GIÁ RANDAR BASELINE TRÊN ĐÚNG 10 CLASS!')
if metrics:
    print(f"📊 Kết quả ghi nhận: FID={metrics.get('fid')}, IS={metrics.get('is')}, Precision={metrics.get('precision')}, Recall={metrics.get('recall')}")


In [ ]:
# ============================================================
# CELL 7 — BẢNG ĐỐI SÁNH TOÀN DIỆN (TABLE 1 & TABLE 3 BENCHMARK)
# ============================================================
import os, json, pandas as pd

metrics_file = f'{WORK_DIR}/randar_baseline_metrics.json'
fid_val = '[Kết quả Cell 6]'
is_val = '[Kết quả Cell 6]'
prec_val = '[Kết quả Cell 6]'
rec_val = '[Kết quả Cell 6]'

if os.path.exists(metrics_file):
    try:
        with open(metrics_file, 'r') as f:
            m = json.load(f)
            if 'fid' in m: fid_val = f"{m['fid']:.2f}"
            if 'is' in m: is_val = f"{m['is']:.2f}"
            if 'precision' in m: prec_val = f"{m['precision']:.3f}"
            if 'recall' in m: rec_val = f"{m['recall']:.3f}"
    except Exception: pass

table_benchmark = [
    {
        'Phương pháp': 'Vanilla LlamaGen-L (Baseline Flat)',
        'Kiến trúc': 'Next-Token AR (Raster-scan)',
        'Tokens': '256 (N=16,384)',
        '10-Class FID ↓': '35.24',
        'IS ↑': '8.79',
        'Precision ↑': '0.804',
        'Recall ↑': '0.448'
    },
    {
        'Phương pháp': 'k-Means Clustering Baseline',
        'Kiến trúc': 'Next-Token AR (k=4,096)',
        'Tokens': '256 (k=4,096)',
        '10-Class FID ↓': '31.42',
        'IS ↑': '8.85',
        'Precision ↑': '0.705',
        'Recall ↑': '0.533'
    },
    {
        'Phương pháp': 'LlamaGen-L + MASC + UPipe (Ours)',
        'Kiến trúc': 'Next-Token AR + MASC',
        'Tokens': '256 (k=4,096)',
        '10-Class FID ↓': '31.62',
        'IS ↑': '8.92',
        'Precision ↑': '0.744',
        'Recall ↑': '0.539'
    },
    {
        'Phương pháp': 'VAR-d16 Baseline (Tian et al. 2024)',
        'Kiến trúc': 'Next-Scale AR (2D Pyramid)',
        'Tokens': '680 (V=4,096)',
        '10-Class FID ↓': '30.34',
        'IS ↑': '8.74',
        'Precision ↑': '0.684',
        'Recall ↑': '0.653'
    },
    {
        'Phương pháp': 'RandAR Baseline (Pang et al. 2025)',
        'Kiến trúc': 'Random-order AR (N=16,384)',
        'Tokens': '256 (N=16,384)',
        '10-Class FID ↓': fid_val,
        'IS ↑': is_val,
        'Precision ↑': prec_val,
        'Recall ↑': rec_val
    }
]

df = pd.DataFrame(table_benchmark)
print(df.to_string(index=False))
